In [ ]:
# 1. Installazione dipendenze Python
!pip install -q onnx onnxruntime onnxscript

import os
import shutil

# =====================================================================
# CONFIGURAZIONE PERCORSI  (unico punto da adattare)
# =====================================================================
# Su Colab i file di lavoro stanno su Drive; in locale nella cartella
# del notebook. I sorgenti C++ vengono presi da ./datagen se presenti
# (come nel repository), altrimenti da BASE_DIR.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/sequence"   # <-- adatta al tuo path Drive
else:
    BASE_DIR = os.getcwd()

NET_NAME = "sequence_net.onnx"          # rete usata da datagen e dal sito
WEIGHTS_NAME = "sequence_net_best.pth"  # checkpoint PyTorch

MODEL_ONNX = os.path.join(BASE_DIR, NET_NAME)
MODEL_PTH = os.path.join(BASE_DIR, WEIGHTS_NAME)
SRC_DIR = "datagen" if os.path.isdir("datagen") else BASE_DIR

print(f"Colab: {IN_COLAB} | BASE_DIR: {BASE_DIR} | sorgenti C++: {SRC_DIR}")

# 2. Scarica ed estrai ONNX Runtime C++ ufficiale (v1.18.0) per Linux x64
if not os.path.exists("onnxruntime-linux-x64-1.18.0"):
    print("Download librerie C++ ONNX Runtime 1.18.0...")
    !wget -q https://github.com/microsoft/onnxruntime/releases/download/v1.18.0/onnxruntime-linux-x64-1.18.0.tgz
    !tar -xzf onnxruntime-linux-x64-1.18.0.tgz

# 3. Porta sorgenti e modelli nella cartella di lavoro
for f in ["datagen.cpp", "match.cpp", "ai.hpp", "board.hpp", "types.hpp"]:
    src = os.path.join(SRC_DIR, f)
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(f"./{f}"):
        shutil.copy(src, f"./{f}")

for f in [NET_NAME, WEIGHTS_NAME]:
    src = os.path.join(BASE_DIR, f)
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(f"./{f}"):
        shutil.copy(src, f"./{f}")

# 4. Compilazione nativa di datagen
print("\n--- Compilazione C++ di datagen ---")
!g++ -O3 -std=c++17 datagen.cpp -o datagen \
    -I./onnxruntime-linux-x64-1.18.0/include \
    -L./onnxruntime-linux-x64-1.18.0/lib \
    -Wl,-rpath,./onnxruntime-linux-x64-1.18.0/lib \
    -lonnxruntime -lpthread

!chmod +x ./datagen

# 4b. Compilazione di match, il giudice usato per il gating
!g++ -O3 -std=c++17 match.cpp -o match \\
    -I./onnxruntime-linux-x64-1.18.0/include \\
    -L./onnxruntime-linux-x64-1.18.0/lib \\
    -Wl,-rpath,./onnxruntime-linux-x64-1.18.0/lib \\
    -lonnxruntime -lpthread

!chmod +x ./match

# 5. Configurazione ambiente librerie
ort_lib_dir = os.path.abspath("./onnxruntime-linux-x64-1.18.0/lib")
os.environ["LD_LIBRARY_PATH"] = f"{ort_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"

# 6. Smoke test (1 partita)
print("\n--- Smoke test datagen ---")
!./datagen 1 1 ./{NET_NAME}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 10.4 MB/s eta 0:00:00
Mounted at /content/drive
Download librerie C++ ONNX Runtime 1.18.0...

--- Compilazione C++ di datagen ---

--- Smoke test datagen ---
Avvio SELF-PLAY NEURALE V2.
Partite totali: 1 | Workers: 1 | Modello: ./sequence_net_v2_single.onnx

[Worker 0] Partito usando ONNX: ./sequence_net_v2_single.onnx

Self-Play Neurale terminato con successo!


In [ ]:
import onnx
# BASE_DIR, MODEL_ONNX e MODEL_PTH arrivano dalla cella 1 (eseguila prima)


# Carica il modello
model = onnx.load(MODEL_ONNX)

# Mostra le informazioni sugli input
print("Input del modello:")
for inp in model.graph.input:
    print(inp.name, inp.type)

# Mostra le informazioni sugli output
print("\nOutput del modello:")
for out in model.graph.output:
    print(out.name, out.type)


Input del modello:
board_input tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 3
    }
    dim {
      dim_value: 10
    }
    dim {
      dim_value: 10
    }
  }
}

hand_input tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 52
    }
  }
}


Output del modello:
policy_output tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 200
    }
  }
}

value_output tensor_type {
  elem_type: 1
  shape {
    dim {
      dim_param: "batch_size"
    }
    dim {
      dim_value: 1
    }
  }
}



In [ ]:
import os
import glob
import time
import subprocess
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import onnx

# =====================================================================
# 1. CONFIGURAZIONE DEL LOOP DI RL
# =====================================================================
# BASE_DIR, MODEL_ONNX e MODEL_PTH arrivano dalla cella 1 (eseguila prima)
DATAGEN_BIN = "./datagen"

# Iperparametri RL
RL_ITERATIONS = 20
GAMES_PER_ITER = 2000
NUM_WORKERS = 4
BATCH_SIZE = 512
EPOCHS_PER_ITER = 3
LR = 1e-4

# Quante iterazioni di partite tenere in memoria per l'addestramento.
# Allenarsi solo sulle ultime 2000 partite fa dimenticare alla rete quello che
# aveva imparato prima e la fa adattare troppo ai dati piu' recenti.
REPLAY_ITERS = 5

# Gating: la rete nuova sostituisce la migliore solo se la batte davvero.
# Senza questo controllo una regressione finisce dritta in produzione.
GATING_GAMES = 300        # mazzi; ogni mazzo si gioca due volte (colori invertiti)
GATING_THRESHOLD = 0.55   # punteggio minimo per promuovere la sfidante
MATCH_BIN = "./match"

# =====================================================================
# 2. ARCHITETTURA RETE NEURALE
# =====================================================================
# Due varianti della stessa rete:
#  - SequenceNetV2: l'originale, con BatchNorm. Da usare partendo da zero.
#  - SequenceNetFolded: senza BatchNorm, con le conv dotate di bias. E' la
#    forma in cui i pesi si recuperano da un .onnx gia' esportato (l'export
#    fonde i BatchNorm dentro le convoluzioni), quindi e' quella da usare per
#    riprendere da una rete di cui si e' perso il checkpoint.
#    Vedi recover_weights.py.
RESUME_FROM_ONNX = True   # False = si riparte da zero con SequenceNetV2

class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

class SequenceNetV2(nn.Module):
    def __init__(self, num_res_blocks=4, channels=64):
        super(SequenceNetV2, self).__init__()
        self.conv_in = nn.Conv2d(3, channels, kernel_size=3, padding=1, bias=False)
        self.bn_in = nn.BatchNorm2d(channels)
        self.res_blocks = nn.ModuleList([ResBlock(channels) for _ in range(num_res_blocks)])

        self.policy_conv = nn.Conv2d(channels, 2, kernel_size=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(2)
        self.policy_fc = nn.Linear(200, 200)

        self.value_conv = nn.Conv2d(channels, 1, kernel_size=1, bias=False)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(100 + 52, 128)
        self.value_fc2 = nn.Linear(128, 1)

    def forward(self, board_tensor, hand_tensor):
        x = F.relu(self.bn_in(self.conv_in(board_tensor)))
        for block in self.res_blocks:
            x = block(x)

        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        policy = self.policy_fc(p)

        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = torch.cat([v, hand_tensor], dim=1)
        v = F.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))

        return policy, value

# =====================================================================
# 3. DATASET BINARIO
# =====================================================================
class ResBlockFolded(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=True)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=True)

    def forward(self, x):
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        return F.relu(out + x)

class SequenceNetFolded(nn.Module):
    def __init__(self, num_res_blocks=4, channels=64):
        super().__init__()
        self.conv_in = nn.Conv2d(3, channels, 3, padding=1, bias=True)
        self.res_blocks = nn.ModuleList([ResBlockFolded(channels) for _ in range(num_res_blocks)])
        self.policy_conv = nn.Conv2d(channels, 2, 1, bias=True)
        self.policy_fc = nn.Linear(200, 200)
        self.value_conv = nn.Conv2d(channels, 1, 1, bias=True)
        self.value_fc1 = nn.Linear(100 + 52, 128)
        self.value_fc2 = nn.Linear(128, 1)

    def forward(self, board_tensor, hand_tensor):
        x = F.relu(self.conv_in(board_tensor))
        for b in self.res_blocks:
            x = b(x)
        p = F.relu(self.policy_conv(x)).view(x.size(0), -1)
        policy = self.policy_fc(p)
        v = F.relu(self.value_conv(x)).view(x.size(0), -1)
        v = torch.cat([v, hand_tensor], dim=1)
        v = F.relu(self.value_fc1(v))
        return policy, torch.tanh(self.value_fc2(v))

class MemoryBufferDataset(Dataset):
    VALID_RESULTS = (-1, 0, 1)

    def __init__(self, data_records):
        self.data = data_records
        self.mask64 = np.array([1 << i for i in range(64)], dtype=np.uint64)
        self.mask36 = np.array([1 << i for i in range(36)], dtype=np.uint64)

    def __len__(self):
        return len(self.data)

    def _unpack_board(self, lo, hi):
        lo_bits = (lo & self.mask64) > 0
        hi_bits = (hi & self.mask36) > 0
        return np.concatenate([lo_bits, hi_bits]).astype(np.float32).reshape(10, 10)

    def __getitem__(self, idx):
        record = self.data[idx]
        my_board = self._unpack_board(record['my_board_lo'], record['my_board_hi'])
        opp_board = self._unpack_board(record['opp_board_lo'], record['opp_board_hi'])
        playable = self._unpack_board(record['playable_lo'], record['playable_hi'])

        board_tensor = np.stack([my_board, opp_board, playable])
        hand_tensor = np.zeros(52, dtype=np.float32)
        hand_size = min(int(record['hand_size']), 7)
        for i in range(hand_size):
            card_id = record['hand'][i]
            if 0 <= card_id <= 51:
                hand_tensor[card_id] += 1.0

        policy_target = int(record['move_pos']) + (100 * int(record['is_removal']))
        value_target = np.array([record['result']], dtype=np.float32)

        return (
            torch.from_numpy(board_tensor),
            torch.from_numpy(hand_tensor),
            torch.tensor(policy_target, dtype=torch.long),
            torch.from_numpy(value_target)
        )

RECORD_DTYPE = np.dtype([
    ('my_board_lo', np.uint64), ('my_board_hi', np.uint64),
    ('opp_board_lo', np.uint64), ('opp_board_hi', np.uint64),
    ('playable_lo', np.uint64), ('playable_hi', np.uint64),
    ('hand', np.int8, (7,)), ('hand_size', np.int8),
    ('move_pos', np.int8), ('is_removal', np.int8), ('result', np.int8)
])

def run_self_play(games, workers, model_onnx_path):
    print(f"\n[SELF-PLAY] Generazione di {games} partite...")
    cmd = [DATAGEN_BIN, str(games), str(workers), model_onnx_path]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.returncode != 0:
        print(res.stdout)
        raise RuntimeError(f"Errore in datagen (exit {res.returncode})")

    files = glob.glob("dataset_worker_*.bin")
    records = []
    for f in files:
        records.append(np.fromfile(f, dtype=RECORD_DTYPE))
        os.remove(f)

    merged = np.concatenate(records)
    np.random.shuffle(merged)
    print(f"[SELF-PLAY] Raccolte {len(merged)} posizioni.")
    return merged

def export_onnx(model, device, onnx_dest_path):
    model.eval()
    dummy_board = torch.randn(1, 3, 10, 10, device=device)
    dummy_hand = torch.randn(1, 52, device=device)
    temp_path = "/tmp/temp_export.onnx"

    torch.onnx.export(
        model,
        (dummy_board, dummy_hand),
        temp_path,
        input_names=["board_input", "hand_input"],
        output_names=["policy_output", "value_output"],
        dynamic_axes={
            "board_input": {0: "batch_size"},
            "hand_input": {0: "batch_size"},
            "policy_output": {0: "batch_size"},
            "value_output": {0: "batch_size"}
        },
        opset_version=18
    )
    onnx_model = onnx.load(temp_path)
    onnx.save_model(onnx_model, onnx_dest_path, save_as_external_data=False)
    if os.path.exists(temp_path):
        os.remove(temp_path)
def run_gating(candidate_onnx, best_onnx, games, workers):
    """Fa giocare la rete candidata contro la migliore attuale.
    Ritorna il punteggio della candidata (1 = vince sempre, 0.5 = pari)."""
    if not os.path.exists(best_onnx):
        return 1.0  # non c'e' ancora un campione: la prima rete passa
    cmd = [MATCH_BIN, str(games), str(workers), candidate_onnx, best_onnx]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.returncode != 0:
        print(res.stdout)
        raise RuntimeError(f"Errore in match (exit {res.returncode})")

    win = loss = draw = 0
    for line in res.stdout.splitlines():
        if "A vince" in line:  win  = int(line.split(":")[1].split("(")[0])
        elif "B vince" in line: loss = int(line.split(":")[1].split("(")[0])
        elif "patte" in line:   draw = int(line.split(":")[1].split("(")[0])
    total = win + loss + draw
    return (win + 0.5 * draw) / total if total else 0.5

# =====================================================================
# 4. LOOP PRINCIPALE
# =====================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"=== LOOP RL SU {device.type.upper()} ===")

    os.makedirs(BASE_DIR, exist_ok=True)
    model = (SequenceNetFolded() if RESUME_FROM_ONNX else SequenceNetV2()).to(device)

    if os.path.exists(MODEL_PTH):
        print(f"Caricamento pesi: {MODEL_PTH}")
        model.load_state_dict(torch.load(MODEL_PTH, map_location=device))
    else:
        print("Inizializzazione pesi da zero...")
        torch.save(model.state_dict(), MODEL_PTH)

    # BEST_ONNX e' la rete campione, quella che genera le partite e che finisce
    # sul sito. MODEL_ONNX e' la candidata prodotta a ogni iterazione.
    BEST_ONNX = os.path.join(BASE_DIR, "sequence_net_best.onnx")
    if not os.path.exists(BEST_ONNX):
        export_onnx(model, device, BEST_ONNX)

    export_onnx(model, device, MODEL_ONNX)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    policy_criterion = nn.CrossEntropyLoss()
    value_criterion = nn.MSELoss()

    # Le partite delle ultime REPLAY_ITERS iterazioni, non solo dell'ultima.
    replay = deque(maxlen=REPLAY_ITERS)

    for iteration in range(1, RL_ITERATIONS + 1):
        print(f"\n--- ITERAZIONE RL {iteration}/{RL_ITERATIONS} ---")
        # Le partite le genera sempre il campione in carica
        replay.append(run_self_play(GAMES_PER_ITER, NUM_WORKERS, BEST_ONNX))
        iter_data = np.concatenate(list(replay))
        np.random.shuffle(iter_data)
        print(f"[BUFFER] {len(iter_data)} posizioni da {len(replay)} iterazioni.")

        dataset = MemoryBufferDataset(iter_data)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=(device.type == "cuda"))

        model.train()
        for epoch in range(EPOCHS_PER_ITER):
            total_loss, correct = 0.0, 0
            for boards, hands, policy_tgts, value_tgts in loader:
                boards, hands = boards.to(device), hands.to(device)
                policy_tgts, value_tgts = policy_tgts.to(device), value_tgts.to(device)

                optimizer.zero_grad()
                p_preds, v_preds = model(boards, hands)

                loss = policy_criterion(p_preds, policy_tgts) + value_criterion(v_preds, value_tgts)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                _, pred_idx = torch.max(p_preds, 1)
                correct += (pred_idx == policy_tgts).sum().item()

            print(f"  [Epoca {epoch+1}/{EPOCHS_PER_ITER}] Loss: {total_loss / len(loader):.4f} | Accuracy: {100.0 * correct / len(dataset):.2f}%")

        torch.save(model.state_dict(), MODEL_PTH)
        export_onnx(model, device, MODEL_ONNX)

        # --- GATING: la candidata entra in produzione solo se batte il campione
        score = run_gating(MODEL_ONNX, BEST_ONNX, GATING_GAMES, NUM_WORKERS)
        print(f"[GATING] La candidata segna {score*100:.1f}% contro il campione "
              f"(soglia {GATING_THRESHOLD*100:.0f}%)")
        if score >= GATING_THRESHOLD:
            export_onnx(model, device, BEST_ONNX)
            torch.save(model.state_dict(), os.path.join(BASE_DIR, "sequence_net_best_accepted.pth"))
            print(" PROMOSSA: e' il nuovo campione.")
        else:
            print(" RESPINTA: il campione resta quello di prima.")

if __name__ == "__main__":
    main()

=== LOOP RL SU CUDA ===
Caricamento pesi: /content/drive/MyDrive/sequence/sequence_net_best.pth


/tmp/ipykernel_3029/786450382.py:156: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SequenceNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SequenceNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

--- ITERAZIONE RL 1/20 ---

[SELF-PLAY] Generazione di 2000 partite...


...

/tmp/ipykernel_3029/786450382.py:156: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SequenceNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SequenceNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
 Modello salvato e ONNX aggiornato.


/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
